# Paired SWE-loss scatter — revised for plotting protocol v5

**Scientific definition preserved at the user's direction (22 September 2026).** This is the notebook's custom analysis: each point is a Noah-grid cell-day where **both** models lose strictly more than 1 mm SWE in one day. The overall window includes the earlier watershed peak date through the later watershed SWE50 date. It is not the workflow's standard April–June H2 event statistic.

The event belongs to its **ending date**; the previous day's state is required, including on the first window date. The preserved categories are [earlier peak, later peak), [later peak, earlier SWE50], and (earlier SWE50, later SWE50]. Equal boundaries can produce empty outer categories. If the later peak exceeds the earlier SWE50, the original three-part interpretation is invalid and execution stops.

SWE50 is each model's first observed daily watershed SWE at or below half of its own available-record peak, reconstructed from Table 3 and checked against native watershed daily means. No smoothing, interpolation, common peak, or alternative spatial averaging is introduced. Daily SWE loss is **−ΔSWE**, not a direct melt or runoff flux. MB = mean(Noah-MP loss − iSnobal loss). r and MB describe paired cell-days; no independent-sample inference is made.

**Run all cells in order** in the v0.8.2 workflow environment. Tables/caches come exclusively from the configured production `output/output1`; caches and weights are never regenerated here. Keep `East_River_fonts` beside the notebooks or set `FONT_PATHS` explicitly. Missing Nimbus Sans raises an error. Paired points are neutral gray; blue/red axis labels retain model identity. A one-series scatter needs no model legend. PNG and PDF exports use new `_v5` stems. Original saved outputs have been cleared.


In [ ]:
from __future__ import annotations
from pathlib import Path
from copy import deepcopy
import math, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from east_river_workflow.config import load_config, WorkflowConfig
from east_river_workflow.data_access import read_noah_grid
from east_river_workflow.grids import load_weights
from east_river_workflow.processing import load_spatial_cache, spatial_cache_path
from east_river_workflow.metrics import pearson_mean_bias


In [ ]:
# Production inputs; all tables, caches and exports use this one output root.
ROOT = Path('/bsuhome/zihaozhang/scratch/iSnobal/East_River_Workflow_Code_v0.8.2')
OUTPUT_DIR = ROOT / 'output' / 'output1'
CONFIG_PATH = ROOT / 'config' / 'east_river_config.yaml'
TABLE3_PATH = OUTPUT_DIR / 'tables' / 'table3_watershed_annual_metrics.csv'
# Optional local font registration; an installed Nimbus Sans also works.
# For Jupyter, set an absolute path here if the working directory differs.
FONT_PATHS = [Path.cwd() / 'East_River_fonts' / 'NimbusSans-Regular.ttf']
MODEL_COLORS = {'iSnobal': '#1F77B4', 'Noah-MP': '#D62728', 'ASO': 'black'}
ABLATION_THRESHOLD_MM_DAY = 1.0
POINT_COLOR = '0.30'
SHOW_STATS = True
FIGURE_STEM = 'h2_postpeak_ablation_all_years_v5'
METRICS_STEM = 'h2_postpeak_ablation_window_metrics_v5'


In [ ]:
def configure_publication_style(font_paths=()):
    for path in font_paths:
        if Path(path).is_file():
            fm.fontManager.addfont(str(path))
    try:
        resolved = fm.findfont(fm.FontProperties(family='Nimbus Sans', weight='normal'),
                               fallback_to_default=False)
    except ValueError as exc:
        raise RuntimeError('Nimbus Sans is required. Install it or set FONT_PATHS to its font file; no fallback is permitted.') from exc
    if fm.FontProperties(fname=resolved).get_name() != 'Nimbus Sans':
        raise RuntimeError(f'Wrong resolved font: {resolved}')
    mpl.rcParams.update({'font.family': 'Nimbus Sans', 'font.size': 18,
        'font.weight': 'normal', 'axes.titlesize': 20, 'axes.titleweight': 'normal',
        'axes.labelweight': 'normal', 'axes.labelsize': 18, 'legend.fontsize': 18,
        'xtick.labelsize': 18, 'ytick.labelsize': 18, 'axes.titlepad': 8,
        'pdf.fonttype': 42, 'ps.fonttype': 42, 'pdf.use14corefonts': False,
        'text.usetex': False, 'axes.grid': False, 'figure.dpi': 120,
        'savefig.dpi': 600, 'lines.linewidth': 2.5})
    print(f'Verified Nimbus Sans: {resolved}')
    return resolved


def validate_table3(frame, years, analysis_start, analysis_end):
    required = {'water_year', 'peak_date_I', 'peak_date_N', 'days_to_50_I',
                'days_to_50_N', 'Delta_t50_days'}
    missing = required - set(frame)
    if missing:
        raise ValueError(f'Table 3 missing columns: {sorted(missing)}')
    years_numeric = np.asarray(years, dtype=float)
    if (not len(years_numeric) or not np.isfinite(years_numeric).all()
            or not np.equal(years_numeric, np.floor(years_numeric)).all()
            or len(set(years_numeric)) != len(years_numeric)):
        raise ValueError('Configured water years must be nonempty, unique integers.')
    years = [int(y) for y in years_numeric]
    frame = frame.copy()
    wy = pd.to_numeric(frame['water_year'], errors='raise')
    if not np.isfinite(wy).all() or not (wy == np.floor(wy)).all():
        raise ValueError('Invalid water_year values in Table 3.')
    frame['water_year'] = wy.astype(int)
    extras = sorted(set(frame['water_year']) - set(years))
    if extras:
        print(f'Table 3 years outside the configured selection: {extras}')
    frame = frame[frame['water_year'].isin(years)].copy()
    if frame['water_year'].duplicated().any():
        raise ValueError('Duplicate Table 3 rows for a selected water year.')
    missing_years = set(years) - set(frame['water_year'])
    if missing_years:
        raise ValueError(f'Table 3 missing water years: {sorted(missing_years)}')
    frame = frame.set_index('water_year').loc[years].reset_index()
    for source in ['I', 'N']:
        col = f'peak_date_{source}'
        dates = pd.to_datetime(frame[col], errors='raise')
        if dates.isna().any() or dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
            raise ValueError(f'{col} must contain nonmissing, timezone-naive daily dates.')
        frame[col] = dates
        duration = pd.to_numeric(frame[f'days_to_50_{source}'], errors='raise')
        if (not np.isfinite(duration).all() or (duration < 0).any()
                or not duration.eq(np.floor(duration)).all()):
            raise ValueError(f'days_to_50_{source} must be finite nonnegative whole days; an unreached SWE50 is undefined, not zero.')
        frame[f'days_to_50_{source}'] = duration.astype(int)
        frame[f'swe50_date_{source}'] = dates + pd.to_timedelta(duration, unit='D')
        for idx, row in frame.iterrows():
            lo = max(pd.Timestamp(int(row.water_year)-1, 10, 1), pd.Timestamp(analysis_start).normalize())
            hi = min(pd.Timestamp(int(row.water_year), 9, 30), pd.Timestamp(analysis_end).normalize())
            if not lo <= row[col] <= row[f'swe50_date_{source}'] <= hi:
                raise ValueError(f'WY {row.water_year}: {source} peak/SWE50 outside the water year or analysis period.')
    official = pd.to_numeric(frame['Delta_t50_days'], errors='raise').to_numpy(float)
    calculated = (frame['days_to_50_N'] - frame['days_to_50_I']).to_numpy(float)
    if not np.isfinite(official).all() or not np.allclose(official, calculated, rtol=0, atol=1e-10):
        raise ValueError('Table 3 Delta_t50_days disagrees with days_to_50_N minus days_to_50_I (or is missing).')
    frame['Delta_t50_days'] = official
    return frame


def validate_daily_timing(table3, table_dir):
    """Check Table 3 against native watershed means, never means of regridded cells."""
    audits = []
    for source, filename in [('I', 'isnobal_watershed_daily.csv'), ('N', 'noahmp_watershed_daily.csv')]:
        path = Path(table_dir) / filename
        daily = pd.read_csv(path)
        if not {'date', 'water_year', 'swe_mm'} <= set(daily):
            raise ValueError(f'Missing date, water_year or swe_mm in {path}')
        daily['date'] = pd.to_datetime(daily['date'], errors='raise')
        if (daily['date'].isna().any() or daily['date'].dt.tz is not None
                or not daily['date'].eq(daily['date'].dt.normalize()).all()
                or daily['date'].duplicated().any()):
            raise ValueError(f'Invalid/duplicate daily dates: {path}')
        expected_wy = daily['date'].dt.year + (daily['date'].dt.month >= 10).astype(int)
        if not pd.to_numeric(daily['water_year'], errors='raise').eq(expected_wy).all():
            raise ValueError(f'Water-year/date disagreement: {path}')
        daily['swe_mm'] = pd.to_numeric(daily['swe_mm'], errors='raise')
        for _, row in table3.iterrows():
            s = daily.loc[expected_wy.eq(row.water_year)].set_index('date')['swe_mm'].sort_index()
            finite = s[np.isfinite(s)]
            if finite.empty or finite.max() <= 0:
                raise ValueError(f'No positive finite peak: {path}, WY {row.water_year}')
            peak_date, peak = finite.idxmax(), float(finite.max())
            post = finite.loc[peak_date:]
            hits = post.index[(post / peak) <= 0.5]
            if not len(hits):
                raise ValueError(f'SWE50 not observed: {path}, WY {row.water_year}')
            swe50 = hits[0]
            if peak_date != row[f'peak_date_{source}'] or swe50 != row[f'swe50_date_{source}']:
                raise ValueError(f'Table 3 and daily watershed SWE disagree: source {source}, WY {row.water_year}')
            peak_col = f'peak_SWE_{source}_mm'
            if peak_col in table3 and not np.isclose(float(row[peak_col]), peak, rtol=1e-9, atol=1e-6):
                raise ValueError(f'Table 3 peak magnitude disagrees: source {source}, WY {row.water_year}')
            expected = pd.date_range(s.index.min(), s.index.max(), freq='D')
            n_missing = int((~np.isfinite(s.reindex(expected))).sum())
            if n_missing:
                warnings.warn(f'{source} WY {row.water_year}: {n_missing} unavailable daily states; crossing is first observed <=50%, with no interpolation.')
            audits.append({'water_year': int(row.water_year), 'source': source,
                'record_start': s.index.min(), 'record_end': s.index.max(),
                'peak_swe_mm': peak, 'peak_date': peak_date, 'swe50_date': swe50,
                'unavailable_daily_states': n_missing, 'table3_agrees': True})
    return pd.DataFrame(audits)


def file_record(path):
    path = Path(path)
    stat = path.stat()
    record = {'path': str(path.resolve()), 'size_bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns}
    if stat.st_size < 20_000_000:
        record['sha256'] = hashlib.sha256(path.read_bytes()).hexdigest()
    return record


def save_publication_figure(fig, directory, stem):
    # Check actual text objects too: rcParams alone cannot prevent explicit overrides.
    for item in fig.findobj(mpl.text.Text):
        if item.get_text():
            resolved = fm.findfont(item.get_fontproperties(), fallback_to_default=False)
            if fm.FontProperties(fname=resolved).get_name() != 'Nimbus Sans':
                raise RuntimeError(f'Figure text uses a different font: {item.get_text()}')
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    for extension in ['png', 'pdf']:
        path = directory / f'{stem}.{extension}'
        # PDF rasterized dense scatter is also 600 dpi; axes/text remain vector.
        fig.savefig(path, dpi=600, bbox_inches='tight', pad_inches=0.05, facecolor='white')
        print(f'Saved: {path}')


In [ ]:
def get_h2_dates(row: pd.Series) -> dict[str, pd.Timestamp]:
    """
    Derive each model's peak and SWE50 dates and the four boundaries used
    to partition the broad earlier-peak -> later-SWE50 interval.
    """

    peak_i = pd.Timestamp(row["peak_date_I"]).normalize()
    peak_n = pd.Timestamp(row["peak_date_N"]).normalize()

    if pd.isna(peak_i) or pd.isna(peak_n):
        raise ValueError("Missing peak date.")

    if not np.isfinite(row["days_to_50_I"]):
        raise ValueError("Missing days_to_50_I.")

    if not np.isfinite(row["days_to_50_N"]):
        raise ValueError("Missing days_to_50_N.")

    swe50_i = (
        peak_i
        + pd.to_timedelta(row["days_to_50_I"], unit="D")
    ).normalize()

    swe50_n = (
        peak_n
        + pd.to_timedelta(row["days_to_50_N"], unit="D")
    ).normalize()

    earlier_peak = min(peak_i, peak_n)
    later_peak = max(peak_i, peak_n)

    earlier_swe50 = min(swe50_i, swe50_n)
    later_swe50 = max(swe50_i, swe50_n)

    # The requested three-window interpretation requires this order.
    # Do not silently allow overlapping categories.
    if later_peak > earlier_swe50:
        raise ValueError(
            "Window ordering is invalid:\n"
            f"  earlier peak  = {earlier_peak.date()}\n"
            f"  later peak    = {later_peak.date()}\n"
            f"  earlier SWE50 = {earlier_swe50.date()}\n"
            f"  later SWE50   = {later_swe50.date()}\n"
            "The later peak occurs after the earlier SWE50, so the "
            "three requested windows would overlap."
        )

    return {
        "peak_i": peak_i,
        "peak_n": peak_n,
        "swe50_i": swe50_i,
        "swe50_n": swe50_n,
        "earlier_peak": earlier_peak,
        "later_peak": later_peak,
        "earlier_swe50": earlier_swe50,
        "later_swe50": later_swe50,
    }

def aligned_spatial_window(ic, nc, start, end):
    expected = pd.date_range(start - pd.Timedelta(days=1), end, freq='D')
    indices = []
    for name, cache in [('iSnobal', ic), ('Noah-MP', nc)]:
        dates = pd.DatetimeIndex(cache['dates'])
        if (dates.hasnans or dates.tz is not None or not dates.equals(dates.normalize())
                or dates.has_duplicates or not dates.is_monotonic_increasing):
            raise ValueError(f'{name}: cache dates must be unique, sorted, timezone-naive daily dates.')
        if np.shape(cache['swe_mm'])[0] != len(dates):
            raise ValueError(f'{name}: cache date/array length mismatch.')
        ix = dates.get_indexer(expected)
        if (ix < 0).any():
            raise ValueError(f'{name}: missing daily states, including required start-1: {expected[ix < 0].strftime("%Y-%m-%d").tolist()}. Do not difference across gaps.')
        indices.append(ix)
    return expected, indices[0], indices[1]

def get_ablation_points(
    ic: dict,
    nc: dict,
    ii: np.ndarray,
    ni: np.ndarray,
    common_dates: pd.DatetimeIndex,
    watershed_mask: np.ndarray,
    dates: dict[str, pd.Timestamp],
    threshold: float,
):
    """
    Calculate all paired cell-day SWE-loss points.

    Event condition:
        dSWE_iSnobal < -threshold
        AND
        dSWE_NoahMP < -threshold

    Returned x/y values are positive SWE-loss magnitudes:
        x = -dSWE_iSnobal
        y = -dSWE_NoahMP
    """

    swe_i = np.asarray(ic["swe_mm"][ii], dtype=float)
    swe_n = np.asarray(nc["swe_mm"][ni], dtype=float)

    if swe_i.shape != swe_n.shape:
        raise ValueError(
            f"Spatial-cache shapes differ: "
            f"{swe_i.shape} vs {swe_n.shape}"
        )

    if len(common_dates) < 2:
        raise ValueError("Not enough common dates to calculate dSWE.")

    if swe_i.ndim != 3 or swe_i.shape[1:] != np.shape(watershed_mask):
        raise ValueError('Expected matching [time,row,col] SWE and [row,col] watershed mask.')
    if swe_i.shape[0] != len(common_dates) or not ((common_dates[1:] - common_dates[:-1]) == pd.Timedelta(days=1)).all():
        raise ValueError('Daily SWE loss requires consecutive one-day intervals.')
    if not np.isfinite(threshold) or threshold < 0:
        raise ValueError('Ablation threshold must be finite and nonnegative.')

    # -------------------------------------------------------------
    # Daily SWE change
    # -------------------------------------------------------------
    dswe_i = np.diff(swe_i, axis=0)
    dswe_n = np.diff(swe_n, axis=0)

    # Positive values plotted as SWE-loss magnitude.
    loss_i = -dswe_i
    loss_n = -dswe_n

    # Each dSWE is assigned to its ending date.
    event_dates = common_dates[1:]

    # -------------------------------------------------------------
    # Strict AND ablation definition
    #
    # dSWE_i < -1 AND dSWE_n < -1
    # -------------------------------------------------------------
    valid_event = (
        np.isfinite(loss_i)
        & np.isfinite(loss_n)
        & (loss_i > threshold)
        & (loss_n > threshold)
        & np.broadcast_to(
            np.asarray(watershed_mask, dtype=bool),
            loss_i.shape,
        )
    )

    # -------------------------------------------------------------
    # Requested temporal categories
    # -------------------------------------------------------------
    before_dates = (
        (event_dates >= dates["earlier_peak"])
        & (event_dates < dates["later_peak"])
    )

    common_dates_mask = (
        (event_dates >= dates["later_peak"])
        & (event_dates <= dates["earlier_swe50"])
    )

    after_dates = (
        (event_dates > dates["earlier_swe50"])
        & (event_dates <= dates["later_swe50"])
    )

    overall_dates = (
        (event_dates >= dates["earlier_peak"])
        & (event_dates <= dates["later_swe50"])
    )

    date_masks = {
        "Before common window": before_dates,
        "Common post-peak window": common_dates_mask,
        "After common window": after_dates,
        "Overall (earlier peak to later SWE50)": overall_dates,
    }

    points = {}

    for category, one_d_mask in date_masks.items():

        time_mask = np.broadcast_to(
            one_d_mask[:, None, None],
            loss_i.shape,
        )

        keep = valid_event & time_mask

        points[category] = (
            loss_i[keep],
            loss_n[keep],
        )

    return points

def calculate_metrics(
    wy: int,
    dates: dict[str, pd.Timestamp],
    points: dict,
) -> pd.DataFrame:

    ranges = {
        "Before common window": (
            dates["earlier_peak"],
            dates["later_peak"] - pd.Timedelta(days=1),
        ),
        "Common post-peak window": (
            dates["later_peak"],
            dates["earlier_swe50"],
        ),
        "After common window": (
            dates["earlier_swe50"] + pd.Timedelta(days=1),
            dates["later_swe50"],
        ),
        "Overall (earlier peak to later SWE50)": (
            dates["earlier_peak"],
            dates["later_swe50"],
        ),
    }

    rows = []

    for category in [
        "Before common window",
        "Common post-peak window",
        "After common window",
        "Overall (earlier peak to later SWE50)",
    ]:

        x, y = points[category]

        stats = pearson_mean_bias(x, y)

        start_date, end_date = ranges[category]

        rows.append(
            {
                "water_year": int(wy),
                "window": category,
                "start_date": start_date if start_date <= end_date else pd.NaT,
                "end_date": end_date if start_date <= end_date else pd.NaT,
                "n_calendar_days": max(0, (end_date-start_date).days+1),
                "event_rule": "both models loss > 1 mm/day",
                "bias_definition": "Noah-MP loss minus iSnobal loss",
                "n_cell_days": int(stats["n"]),
                "pearson_r": stats["pearson_r"],
                # Noah-MP loss minus iSnobal loss
                "mean_bias_mm_day": stats["mean_bias"],
            }
        )

    return pd.DataFrame(rows)


In [ ]:
cfg_original = load_config(CONFIG_PATH)
data = deepcopy(cfg_original.data)
if cfg_original.output_dir.resolve() != OUTPUT_DIR.resolve():
    print(f'Config output_dir is {cfg_original.output_dir}; using the user-specified production root: {OUTPUT_DIR}')
data['paths']['output_dir'] = str(OUTPUT_DIR)
cfg = WorkflowConfig(data=data, source_path=cfg_original.source_path)
years = cfg.section('project')['water_years']
table3 = validate_table3(pd.read_csv(TABLE3_PATH), years, cfg.analysis_start, cfg.analysis_end)
years = table3['water_year'].tolist()
timing_audit = validate_daily_timing(table3, OUTPUT_DIR / 'tables')
font_file = configure_publication_style(FONT_PATHS)
print(timing_audit.to_string(index=False))
print('Peak and SWE50 refer to the available analysis record; the final WY may be partial.')
input_records = [file_record(p) for p in [CONFIG_PATH, TABLE3_PATH,
    OUTPUT_DIR/'tables/isnobal_watershed_daily.csv', OUTPUT_DIR/'tables/noahmp_watershed_daily.csv']]


In [ ]:
noah_grid, _, _, _, _ = read_noah_grid(
    cfg.data["paths"]["noahmp_geo_file"],
    cfg.section("noahmp"),
)

weights_path = (
    cfg.output_dir
    / "cache"
    / "east_river_fraction_noahmp.npz"
)

if not weights_path.exists():
    raise FileNotFoundError(
        "Existing Noah-MP watershed-weight cache was not found:\n"
        f"{weights_path}\n\n"
        "This script intentionally does not regenerate it."
    )

noah_weights = load_weights(
    weights_path,
    noah_grid,
)

# Existing H1/H2 convention:
# retain every Noah cell with any East River polygon overlap.
if (not np.isfinite(noah_weights).all() or (noah_weights < 0).any()
        or (noah_weights > 1).any() or not (noah_weights > 0).any()):
    raise ValueError('Invalid or empty cached watershed fractional weights.')
watershed_mask = noah_weights > 0
input_records.append(file_record(weights_path))
input_records.append(file_record(cfg.data['paths']['noahmp_geo_file']))


# =============================================================================
# EXTRACT ALL YEARS FIRST
#
# Nothing is sampled or thinned.
# Each retained point = one paired Noah-grid cell-day.
# =============================================================================

year_points = {}
metrics_frames = []

for wy in years:

    row_match = table3[
        table3["water_year"].eq(wy)
    ]

    if row_match.empty:
        raise ValueError(
            f"WY {wy} is missing from Table 3."
        )

    row = row_match.iloc[0]

    dates = get_h2_dates(row)

    ic = load_spatial_cache(
        spatial_cache_path(cfg, "isnobal", wy)
    )

    nc = load_spatial_cache(
        spatial_cache_path(cfg, "noahmp", wy)
    )

    input_records.extend(file_record(spatial_cache_path(cfg, source, wy)) for source in ['isnobal', 'noahmp'])
    common, ii, ni = aligned_spatial_window(
        ic,
        nc,
        dates["earlier_peak"],
        dates["later_swe50"],
    )

    points = get_ablation_points(
        ic=ic,
        nc=nc,
        ii=ii,
        ni=ni,
        common_dates=common,
        watershed_mask=watershed_mask,
        dates=dates,
        threshold=ABLATION_THRESHOLD_MM_DAY,
    )

    # Only OVERALL points are plotted.
    x_overall, y_overall = points[
        "Overall (earlier peak to later SWE50)"
    ]

    year_points[wy] = {
        "x": x_overall,
        "y": y_overall,
    }

    metrics_frames.append(
        calculate_metrics(
            wy=wy,
            dates=dates,
            points=points,
        )
    )


In [ ]:
all_values = []

for wy in years:
    all_values.append(year_points[wy]["x"])
    all_values.append(year_points[wy]["y"])

nonempty = [
    a[np.isfinite(a)]
    for a in all_values
    if len(a) > 0
]

if nonempty:
    global_max = float(
        np.max(
            np.concatenate(nonempty)
        )
    )
else:
    global_max = 10.0

# Round upward to a clean 5 mm/day boundary.
axis_max = max(
    5.0,
    math.ceil(global_max * 1.04 / 5.0) * 5.0,
)


# =============================================================================
# PLOT — ALL YEARS, ONE COLOR
# =============================================================================

ncols = 3
nrows = math.ceil(len(years) / ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(21, 5.5 * nrows),
    sharex=True,
    sharey=True,
    squeeze=False,
)

axes = np.asarray(axes).reshape(-1)

for ax, wy in zip(axes, years):

    x = year_points[wy]["x"]
    y = year_points[wy]["y"]

    # ---------------------------------------------------------
    # Overall statistics: earlier peak -> later SWE50
    # ---------------------------------------------------------
    stats = pearson_mean_bias(x, y)

    r = stats["pearson_r"]
    bias = stats["mean_bias"]

    rtxt = "NA" if not np.isfinite(r) else f"{r:.2f}"
    btxt = "NA" if not np.isfinite(bias) else f"{bias:+.2f}"

    # ---------------------------------------------------------
    # Scatter
    # ---------------------------------------------------------
    ax.scatter(
        x,
        y,
        s=5,
        alpha=0.18,
        color=POINT_COLOR,
        edgecolors="none",
        rasterized=True,
        zorder=2,
    )

    # 1:1 line
    ax.plot(
        [0, axis_max],
        [0, axis_max],
        color="0.35",
        lw=1.0,
        zorder=1,
    )

    ax.set_xlim(0, axis_max)
    ax.set_ylim(0, axis_max)

    ax.set_title(
        f"WY {wy}",
        fontsize=20,
        fontweight="normal",
        pad=8,
    )

    # ---------------------------------------------------------
    # Optional descriptive r and MB — upper right
    # ---------------------------------------------------------
    if SHOW_STATS:
        ax.text(
            0.97,
            0.95,
            f"r = {rtxt}\nMB = {btxt} mm/day",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=18,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.92, pad=2),
        )


    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)

    ax.tick_params(
        axis="both",
        which="major",
        width=1.0,
        length=5,
    )


# Hide unused axes if number of years is not divisible by 3.
for ax in axes[len(years):]:
    ax.set_visible(False)


# Shared source labels; neutral points represent pairs, not an individual model.
fig.supxlabel(
    "iSnobal daily SWE loss (mm/day)",
    y=0.095,
    color=MODEL_COLORS["iSnobal"],
)

fig.supylabel(
    "Noah-MP daily SWE loss (mm/day)",
    x=0.015,
    color=MODEL_COLORS["Noah-MP"],
)

fig.subplots_adjust(
    top=0.95,
    bottom=0.15,
    left=0.07,
    right=0.98,
    hspace=0.30,
    wspace=0.16,
)


In [ ]:
save_publication_figure(fig, OUTPUT_DIR/'figures/scientific', FIGURE_STEM)
plt.show()
metrics_table = pd.concat(metrics_frames, ignore_index=True)
# Preserve full numeric precision in the CSV; only panel annotations are rounded.
table_dir = OUTPUT_DIR / 'tables'
table_dir.mkdir(parents=True, exist_ok=True)
metrics_table.to_csv(table_dir/f'{METRICS_STEM}.csv', index=False)
timing_audit.to_csv(table_dir/f'{METRICS_STEM}_timing_audit.csv', index=False)
provenance = {'analysis': 'custom AND-event earlier-peak to later-SWE50 diagnostic',
    'threshold_mm_day': ABLATION_THRESHOLD_MM_DAY,
    'event_date': 'ending date; retain start minus one day',
    'spatial_support': 'unweighted cell-days on Noah grid where cached noah_weights > 0',
    'bias': 'Noah-MP loss minus iSnobal loss', 'font': file_record(font_file), 'inputs': input_records}
(table_dir/f'{METRICS_STEM}_provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print(metrics_table.to_string(index=False))
